In [3]:

from ..Agent import *

#智能体多工具调用+模拟重试(通过系统提示词加全局变量模拟，可能由于内部配置，这里最多重试3次，3次失败就不会再调用了)
#工具调用时，如果调用了工具会先返回带工具结果tool_calls的aiMessage，最后再将结果组装为final_response返回最终的aiMessage

retries=0

@tool
def search_products(product:str):
    '''
    查询对应的商品信息

    Args:
        product:商品名称
    '''
    global retries
    if retries <2:
        retries+=1
        return f'工具调用失败，请重试'
    return f'{product}是一款不错的商品，目前很流行'
@tool
def search_products_stock(product:str):
    '''
    查询对应的商品库存

    Args:
        product:商品名称
    '''
    stock=0
    if product == '手机':
        stock=50
        return f'{product}当前库存余额为{stock}台'
    elif product == '裤子':
        stock=20
        return f'{product}当前库存余额为{stock}条'
    else:
        return f'{product}当前库存余额为{stock}'

load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-pro',
    model_provider='deepseek'
    # extra_body={"thinking":{"type":"disabled"}}
)
#智能体命名后输出的aiMessage带有name,多用于多智能体协作场景
#可以将系统提示词写在智能体内部，不用加在消息列表中
myagent=create_agent(
    model=model,
    tools=[search_products,search_products_stock],
    name='myagent',
    system_prompt='你是一个AI助手，能够合理调用工具回答问题，如果工具结果包含请重试，需要重新调用工具'
)
messages=[
    HumanMessage('帮我查询裤子的商品信息和库存')
]
response=myagent.invoke({
    'messages': messages
}
)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

帮我查询裤子的商品信息和库存
================================== Ai Message ==================================
Name: myagent

好的，我同时为您查询裤子的商品信息和库存情况。
Tool Calls:
  search_products (call_00_KirL2qfDOHSZ1AjWZtY12060)
 Call ID: call_00_KirL2qfDOHSZ1AjWZtY12060
  Args:
    product: 裤子
  search_products_stock (call_01_s9IKsSOur0Cicy3rvmda6221)
 Call ID: call_01_s9IKsSOur0Cicy3rvmda6221
  Args:
    product: 裤子
================================= Tool Message =================================
Name: search_products

工具调用失败，请重试
================================= Tool Message =================================
Name: search_products_stock

裤子当前库存余额为20条
================================== Ai Message ==================================
Name: myagent

商品信息查询失败，正在为您重新查询商品信息。
Tool Calls:
  search_products (call_00_nMTcpptfOcYWNjzUMQfo5569)
 Call ID: call_00_nMTcpptfOcYWNjzUMQfo5569
  Args:
    product: 裤子
================================= To